# The optimization workflow

**Goal.** Run an optimization with APEX from start to finish: copy an example, read its configuration, validate it, submit it, wait for it and read the result. You will run two optimizers on Rosenbrock's function, edit copies of configurations and rerun them, then run two global-search optimizers on Rastrigin's function, where the shape of the problem matters. At the end the same runs are looked at from the terminal.

**Prerequisites.** APEX installed and basic Python. The examples use analytic test functions, so there is no simulator to set up.

**Words used here.**

- *PET* is the toolbox whose optimizers (EnOpt, GenOpt, SmcOpt and others) APEX runs for you. You choose one by name in the configuration.
- *EnOpt* is an *ensemble* method: at each iteration it draws `ne` random samples (an ensemble) of the controls around the current point, evaluates the objective on each, and estimates the gradient from those results. This estimate is the *ensemble gradient*.
- A *trust region* method is deterministic: it works with a local model of the objective and a radius within which it trusts that model.
- *Backtracking* means that when a trial step does not improve the objective enough, the optimizer tries a shorter step instead.

**Where files go.** The notebook uses APEX's Python client on a *services root* of its own: a folder where APEX keeps its jobs and results. Here it is `apex_home`, and example copies go in `work`. Both are relative to the notebook's working folder, so the notebook touches nothing else.

**Two kinds of state.** `client.wait` returns the state of the *job* you submitted. The optimization summary carries the state of the *run*. Both tell you about the work, not about the answer: only the objective value says how close you got to the minimum.

In [1]:
from pathlib import Path


def rel(p):
    """Show a path relative to the working folder (it may already be relative)."""
    p = Path(p)
    try:
        return p.relative_to(Path.cwd())
    except ValueError:
        return p


print("working folder name:", Path.cwd().name)

working folder name: optimization-bd9d591b


## Step 1: copy an example from the catalogue

APEX ships a catalogue of examples. Copying one into a working folder gives you configuration files you can read and edit.

The example `optimization` holds two configurations of the same problem, *Rosenbrock's function minimised by EnOpt and by a trust region*: `rosenbrock_enopt.toml` and `rosenbrock_trust_region.toml`.

In [2]:
import pandas as pd

from apex.client import Client
from apex.services.examples import copy_example, examples_root, find_examples
from apex.services.settings import ExamplesSection

catalogue = {e.id: e for e in find_examples(examples_root(ExamplesSection()))}
copied = copy_example(catalogue["optimization"], Path("work"))
config = copied.folder / "rosenbrock_enopt.toml"
config_trust = copied.folder / "rosenbrock_trust_region.toml"
print(rel(config))
print(rel(config_trust))

work/optimization/rosenbrock_enopt.toml
work/optimization/rosenbrock_trust_region.toml


## Step 2: read the configuration

An optimization is described by one TOML file: the objective, the controls (the variables being optimized) and the optimizer with its options. Print the EnOpt file without its comments to see the settings alone.

In [3]:
def setting(line):
    for index, char in enumerate(line):
        if char == "#" and line[:index].count('"') % 2 == 0:
            return line[:index].rstrip()
    return line.rstrip()


settings = [setting(line) for line in config.read_text().splitlines()]
print("\n".join(line for line in settings if line))

[optimization]
name = "rosenbrock_enopt"
seed = 2026
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
lower = -2.0
upper = 2.0
variance = 0.001
[objective]
name = "rosenbrock"
[optimizer]
name = "enopt"
options = { ne = 20, maxiter = 300, alpha = 0.05, optimizer = "Adam", beta = 0.9, resample = 2 }


How to read it:

- `[optimization]` gives the run a `name` and a `seed`, so a run can be repeated.
- `[[controls.variables]]` declares a variable `x`. Its `initial` list has four values, so the problem has four controls, which the results name `x[0]` to `x[3]`. `lower` and `upper` bound them and `variance` sets the spread of the samples around the current point.
- `[objective]` names the function to minimise, here `rosenbrock`.
- `[optimizer]` names the method and passes `options`. For `enopt`:
  - `ne` is the ensemble size (default 100) and `maxiter` the iteration limit (default 100).
  - `alpha` is the initial step size (default 0.1), `optimizer` the step rule (`GD`, `Adam`, ...) and `beta` the momentum.
  - Backtracking: `alpha_maxiter` (default 5) is the number of shorter trial steps tried before the attempt fails; `resample` (default 0) is how many times a new ensemble is drawn after that.
  - `tol` (optional, not set by default) is the improvement of the mean objective a trial needs; leave it out unless you want to demand a minimum improvement.
  - `ftol`, `xtol` and `gtol` are convergence tolerances: when a change in the objective, in the controls or in the gradient falls below them, the run stops before `maxiter`.

With four controls, Rosenbrock's minimum is at `x = (1, 1, 1, 1)`, where the objective is 0. The function has a single minimum, but it lies at the end of a long, curved, nearly flat valley that is slow to follow. Methods tend to stop somewhere along the valley rather than at its end.

Now the trust-region file. The objective and controls are the same; the optimizer section differs. Not every optimizer takes the same options, so check the reference for the one you choose.

In [4]:
settings_trust = [setting(line) for line in config_trust.read_text().splitlines()]
print("\n".join(line for line in settings_trust if line))

[optimization]
name = "rosenbrock_trust_region"
seed = 1
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
[objective]
name = "rosenbrock"
[optimizer]
name = "trust_region"
options = { maxiter = 100 }


## Step 3: validate before running

Validation checks the configuration and reports a plan, without simulating anything. It is cheap, so make it a habit: a typo in a name shows up here at once instead of after a long run.

The plan has a `max_evaluations` field, the most objective evaluations the plan allows the optimizer. It can be `None`, which means the plan states no figure; the helper below says so.

In [5]:
client = Client(Path("apex_home"))  # a services root of its own


def describe(plan):
    budget = plan.max_evaluations
    shown = "no figure in the plan" if budget is None else f"max_evaluations = {budget}"
    print(f"{plan.optimizer} on {plan.objective}: {plan.n_controls} controls, {shown}")


checked = client.validate_optimization(config)
print("valid:", checked.valid)
plan = checked.optimization
describe(plan)

valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 23402


## Step 4: submit and wait

`client.submit_optimization` takes the configuration and returns a handle that carries a job id. `client.wait` waits on that job, up to the timeout, and returns its status.

What to do with the outcome:

- If the state printed is `complete`, go on to read the result.
- If the wait ends while the job is still running, the job is not lost. Wait again with `client.wait(handle.id, ...)` and a longer timeout.
- For anything else, the code prints the whole status, which carries the details, and then looks for a run with the job's id among `client.optimizations()`. If a row is found, its `path` is the run folder; if none is found, no run was created and the status is all there is to read.

In [6]:
handle = client.submit_optimization(config)
status = client.wait(handle.id, timeout_s=600, poll_s=0.5)
print("job state:", status.state)
if status.state != "complete":
    print(status)
    rows = [r for r in client.optimizations() if r["job"] == handle.id]
    if rows:
        print("run folder:", rel(rows[0]["path"]))
    else:
        print("no run folder was created for this job")

job state: complete


## Step 5: read the result

Find the run of our job by its job id among `client.optimizations()`, and ask for its summary. It holds the run's state, the final objective value and the controls found. Always look at the objective: the states do not tell you whether the minimum was reached.

In [7]:
row = next(r for r in client.optimizations() if r["job"] == handle.id)
summary = client.optimization_summary(Path(row["path"]))
print("run state:", summary.state, "|", summary.optimizer, "on", summary.objective)
print("objective:", summary.result["objective"])
names, values = summary.result["names"], summary.result["x"]
print("controls:", dict(zip(names, values, strict=True)))
print("run folder:", rel(row["path"]))

run state: finished | enopt on rosenbrock
objective: 0.9878013710933521
controls: {'x[0]': 0.6878032981993977, 'x[1]': 0.48472813220308203, 'x[2]': 0.26026533786111106, 'x[3]': 0.0680490417549587}
run folder: apex_home/optimizations/rosenbrock_enopt/fb2db2fdc2f6


Compare with the known answer: the minimum is 0 at `(1, 1, 1, 1)`. If the objective is clearly above 0, the run stopped short of the minimum somewhere along the valley. That is common for Rosenbrock and is not a sign that something went wrong; the function has only one minimum.

A run submitted through the client lives in `apex_home/optimizations/<name>/<id>`; the id differs from run to run.

## Step 6: edit a copy and rerun

To try to get closer, give EnOpt a larger iteration limit, `maxiter`. Keep every run separate: write a copy of the file with a new `name` and a larger `maxiter`, then validate and submit the copy exactly as before. A run already has its own directory, so do not reuse a name.

The code compares the two objectives itself, so you do not have to assume the outcome. It also writes a second copy, `rosenbrock_enopt_cli.toml`, that the terminal section uses.

In [8]:
text = config.read_text()
long_text = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_long"').replace(
    "maxiter = 300", "maxiter = 1000"
)
config_long = copied.folder / "rosenbrock_enopt_long.toml"
config_long.write_text(long_text)

# A second copy for the terminal section below (same settings, its own name).
config_cli = copied.folder / "rosenbrock_enopt_cli.toml"
config_cli.write_text(text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_cli"'))

print("changed:", long_text != text)
checked_long = client.validate_optimization(config_long)
print("valid:", checked_long.valid)
describe(checked_long.optimization)

handle_long = client.submit_optimization(config_long)
status_long = client.wait(handle_long.id, timeout_s=900, poll_s=0.5)
print("job state:", status_long.state)
row_long = next(r for r in client.optimizations() if r["job"] == handle_long.id)
summary_long = client.optimization_summary(Path(row_long["path"]))
before = summary.result["objective"]
after = summary_long.result["objective"]
print("run state:", summary_long.state, "objective:", after)
if after < before:
    print(f"maxiter 1000 improved the objective: {before:.6f} -> {after:.6f}")
elif after == before:
    print("identical objective: the larger limit changed nothing")
else:
    print(f"the objective did not improve: {before:.6f} -> {after:.6f}")

changed: True
valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 78002


job state: complete
run state: finished objective: 0.9878013710933521
identical objective: the larger limit changed nothing


Read the last line. If the objective improved, the extra iterations were used. If it is identical to step 5, the larger `maxiter` made no difference: the first run ended before reaching its limit, because one of the optimizer's own stopping rules (`ftol`, `xtol` or `gtol`) fired. Then a bigger iteration limit is not the way forward; changing other options (for example `alpha`, `ne` or the `seed`) in a renamed copy is. To see how the objective moved step by step, use `apex summary` on the run (terminal section).

## Step 7: run the other optimizer and compare

The same steps work for the trust-region configuration.

In [9]:
checked_trust = client.validate_optimization(config_trust)
print("valid:", checked_trust.valid)
describe(checked_trust.optimization)

handle_trust = client.submit_optimization(config_trust)
status_trust = client.wait(handle_trust.id, timeout_s=600, poll_s=0.5)
print("job state:", status_trust.state)

row_trust = next(r for r in client.optimizations() if r["job"] == handle_trust.id)
summary_trust = client.optimization_summary(Path(row_trust["path"]))

plans = {
    "maxiter 300": plan,
    "maxiter 1000": checked_long.optimization,
    "maxiter 100": checked_trust.optimization,
}
compare = pd.DataFrame(
    [
        {
            "optimizer": s.optimizer,
            "run": label,
            "planned max evaluations": plans[label].max_evaluations,
            "state": s.state,
            "objective": s.result["objective"],
            "controls": [round(float(v), 3) for v in s.result["x"]],
        }
        for label, s in (
            ("maxiter 300", summary),
            ("maxiter 1000", summary_long),
            ("maxiter 100", summary_trust),
        )
    ]
)
print(compare.to_string(index=False))

valid: True
trust_region on rosenbrock: 4 controls, max_evaluations = 502


job state: complete
   optimizer          run  planned max evaluations    state  objective                      controls
       enopt  maxiter 300                    23402 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
       enopt maxiter 1000                    78002 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
trust_region  maxiter 100                      502 finished   3.701429 [-0.776, 0.613, 0.382, 0.146]


Reading the table:

- Compare the `objective` column: a lower value is closer to the minimum 0, under each run's own stopping rules.
- `planned max evaluations` is a ceiling, not what was used. EnOpt evaluates an ensemble at every iteration, so its ceiling is much larger than the trust-region plan's. A run that stops on a tolerance uses less than its ceiling. If a plan printed no figure, the budgets cannot be compared in evaluations from the plans alone.
- The conclusion is modest: the table shows how far each configuration got, not which method is better in general. The runs use different seeds and iteration limits, and one run of a random method says little. For a fair comparison, give the methods the same budget measured the same way and repeat with several seeds.

### Trying a larger `trust_radius`

If the trust-region objective is clearly above 0, it may have stopped early. `trust_region` takes these options (all go in the `options = { ... }` line of `[optimizer]`):

- `trust_radius`: the initial radius of the region where the local model is trusted (default 1.0). `trust_radius_max` and `trust_radius_min` limit it (defaults 100 and 1/1000 of the initial radius); the run stops when the radius reaches the minimum.
- `ftol` (default 1e-5), `xtol` (default 1e-8) and `gtol` (default 1e-5): the tolerances on the objective, the controls and the projected gradient below which the run counts as converged.
- `maxiter` (default 100).

The cell below writes a renamed copy with `trust_radius = 3.0`, runs it and compares the objective.

In [10]:
trust_text = config_trust.read_text()
big_text = trust_text.replace(
    'name = "rosenbrock_trust_region"', 'name = "rosenbrock_trust_region_radius3"'
).replace("options = { maxiter = 100 }", "options = { maxiter = 100, trust_radius = 3.0 }")
config_big = copied.folder / "rosenbrock_trust_region_radius3.toml"
config_big.write_text(big_text)
print("changed:", big_text != trust_text)

checked_big = client.validate_optimization(config_big)
print("valid:", checked_big.valid)
describe(checked_big.optimization)
handle_big = client.submit_optimization(config_big)
status_big = client.wait(handle_big.id, timeout_s=600, poll_s=0.5)
print("job state:", status_big.state)
row_big = next(r for r in client.optimizations() if r["job"] == handle_big.id)
summary_big = client.optimization_summary(Path(row_big["path"]))
print("objective with trust_radius 1.0:", summary_trust.result["objective"])
print("objective with trust_radius 3.0:", summary_big.result["objective"])

changed: True
valid: True
trust_region on rosenbrock: 4 controls, max_evaluations = 502


job state: complete
objective with trust_radius 1.0: 3.701428613683974
objective with trust_radius 3.0: 3.7014286167153996


If the two objectives are equal, the radius was not what ended the first run; look at the tolerances next. Change one option at a time, in a renamed copy, so you know which one made the difference.

## Step 8: when the landscape has many minima

Rosenbrock's difficulty is the slow valley to a single minimum. Other problems have many minima. The `toy/rastrigin` example has a local minimum near every integer point: the population optimizers, the genetic algorithm (`ga`) and particle swarm (`pso`), search the whole range and can find the global one, where an ensemble gradient stops at the nearest. Its configurations are `rastrigin_ga.toml` and `rastrigin_pso.toml`. The workflow is the same, so we wrap the steps in a small function and run both.

In [11]:
rastrigin = copy_example(catalogue["toy/rastrigin"], Path("work"))


def optimize(cfg):
    """Validate, submit and wait for one configuration; return its plan and summary."""
    check = client.validate_optimization(cfg)
    print("valid:", check.valid)
    job = client.submit_optimization(cfg)
    final = client.wait(job.id, timeout_s=600, poll_s=0.5)
    print("job:", final.state)
    found = next(r for r in client.optimizations() if r["job"] == job.id)
    return check.optimization, client.optimization_summary(Path(found["path"]))


config_ga = rastrigin.folder / "rastrigin_ga.toml"
config_pso = rastrigin.folder / "rastrigin_pso.toml"
plan_ga, summary_ga = optimize(config_ga)
plan_pso, summary_pso = optimize(config_pso)

results = pd.DataFrame(
    [
        {
            "optimizer": p.optimizer,
            "max evaluations": p.max_evaluations,
            "objective": s.result["objective"],
            "controls": [round(float(v), 4) for v in s.result["x"]],
        }
        for p, s in ((plan_ga, summary_ga), (plan_pso, summary_pso))
    ]
)
print(results.to_string(index=False))

valid: True


job: complete
valid: True


job: complete
optimizer  max evaluations  objective         controls
       ga             4020   0.001114 [-0.0022, 0.001]
      pso             4020   0.000000      [-0.0, 0.0]


The global minimum of Rastrigin is 0 at the origin. An objective very close to 0 with controls close to `(0, 0)` means the global minimum was found. A run trapped in a local minimum would end near another integer point with an objective of about 1 or more. If both rows are near 0, a small difference between them is the precision each reached, not a different basin, and one run each is too little to rank the two methods.

To try another optimizer on a problem, change only the configuration file you pass to `optimize`.

## The same from the terminal

The commands below run after every cell above has run, from the folder that holds `work` and `apex_home`.

**Validate a configuration.** `apex validate` checks a configuration and plans its cost without simulating anything, as step 3 did. This one is the copy the notebook wrote in step 6 but never ran:

```bash
uv run apex validate work/optimization/rosenbrock_enopt_cli.toml
```

**Summarise runs.** `apex summary` lists every metric at every step of the runs it finds under the folders you give. For an optimization, the steps are the optimization steps; read the objective from the first step to the last:

- A healthy run shows the objective falling quickly at first and then levelling off.
- If it is still falling at the last step, the run stopped early: raise `maxiter` in a renamed copy.
- If it is flat from the start or rises, change the step size (`alpha`) or the seed in a renamed copy.
- Each run is listed under its own directory, and that directory is the `RUN_DIR` that other commands take.

This reads the runs the notebook made through the client:

```bash
uv run apex summary apex_home
```

**Report one run.** `apex report RUN_DIR` prints the metric history of a run and, with `--plots`, also writes figures (PNG and PDF) into `RUN_DIR/report` unless you pass `--output`. Here the run folder is found by listing `apex_home/optimizations/rosenbrock_enopt`, which holds one folder per run, named by its id; there is one run of that name:

```bash
uv run apex report "$(ls -d apex_home/optimizations/rosenbrock_enopt/* | head -n 1)" --plots
```

**Starting and continuing optimizations from the terminal.** The notebook does not run these, but you will meet them:

- `apex optimize CONFIG` starts an optimization directly from a configuration file. `--no-progress` turns off the progress display.
- `--max-iterations N` stops the run after N accepted iterations, as the optimizer counts them. This differs from `maxiter`, which is the optimizer's own limit in the configuration. A run stopped by `--max-iterations` is paused, not finished, and can be continued.
- `apex optimize --resume RUN_DIR` continues the optimization run in `RUN_DIR`. To find that folder, run `apex summary` on the folder that holds the run and use the run directory it lists. Runs made through the client are in `apex_home/optimizations/<name>/<id>`.
- `apex run CONFIG` starts a history-matching run, not an optimization; `apex resume RUN_DIR` continues such a run. Neither is used in this notebook. For an optimization, use `apex optimize --resume`.

To run a configuration a notebook already ran, copy it and give the copy another `name` in `[optimization]`, as in step 6.

## Pitfalls

- **Skipping validation.** A wrong name is caught at once by `validate_optimization`, but only after a whole run if you submit directly.
- **Mixing up the two states.** The job state and the run state both describe the work, not the answer. Judge the objective against what you know of the problem.
- **Assuming more iterations always help.** If a run did not stop at its limit, raising the limit changes nothing, as step 6 checks.
- **Reading one run as proof.** Random methods give different results for different seeds; repeat with several seeds if it matters.
- **Comparing unequal budgets.** A plan can show `None` for the evaluations, and a planned maximum is not the number used. Then compare the objectives reached under stated stopping rules, and say so.
- **Rerunning under the same name.** Copy the configuration and change `name`.
- **Choosing an ensemble-gradient method for a multimodal problem.** On Rastrigin-like problems it stops at the nearest minimum; use a global optimizer, as in step 8.
- **Looking in the wrong place.** Find results by job id through `client.optimizations()`, or list `apex_home/optimizations/<name>`, not by guessing folder names.
- **Running the terminal from another folder.** The relative paths `work` and `apex_home` depend on the folder you are in.

## What to read next

The examples catalogue has more optimization problems that follow the same workflow, including a family of multi-objective test problems (`toy/multi_objective/...`) solved by NSGA-II, NSGA-III and MOEA/D, and two toy-reservoir examples (`toy/toy_reservoir_npv_water` and `toy/toy_reservoir_robust_npv`) that trade NPV against water injected, or against its spread over model realizations. Copy one with `copy_example`, read its TOML as in step 2, and follow the same steps. The reference pages for the `enopt`, `trust_region`, `ga` and `pso` optimizers list every option with its default.